# 第47课 · 亲手搭建 log-Mel 流水线 — log_mel_spectrogram 从公式到 NumPy，与仓库输出对齐

**目标**：从零搭建 log-mel 流水线：STFT → 功率谱（power spectrum） → Mel 滤波 → log 压缩，得到 `(n_frames, n_mels)` 矩阵。

🔗 Aurora 连接：`aurora.audio.mel.mel_spectrogram()` — Month 2 关键词识别分类器的输入特征。

← **上一课**　[L46 · Mel 频率尺度](L46_mel.ipynb)

> 上节课学习了 **Mel 频率尺度**：人耳对数感知，mel = 2595·log₁₀(1+f/700)，三角滤波器。  
> 本课将探讨 **亲手写 Mel 滤波器**。

## 学习目标

1. 能从功率谱手推 Mel 能量矩阵：理解 `power @ fb.T` 的 shape 流向
2. 能解释 eps 的数值意义：为何 log(1e-8) ≈ -18.4 是可接受的下界
3. 能独立实现 `log_mel_spectrogram` 并与 `aurora.audio.mel` 参考输出对齐至 1e-6
4. 能解释 Mel 尺度的非线性分布：为何前 40 个 bin 覆盖 0–1730 Hz，后 40 个 bin 覆盖 1730–8000 Hz


## 本课剧情：为什么语音识别用 log-Mel，而不是原始波形？

你跟 Siri 说"播放音乐"，它识别的不是麦克风录到的原始 PCM 数字。原始波形包含 16000 个采样点/秒，但 99% 是相位噪声——对识别毫无帮助。

语音识别的真正输入是 **log-Mel 频谱**：先把信号压缩到人耳感知频率，再用 log 压缩响度，把 80 维向量/帧送进神经网络。这是 Whisper、Wav2Vec2、所有主流 ASR 模型的标准输入格式。

**完整 5 步流水线**：

```
x          (T,)            ← 原始音频
↓ STFT
S          (n_frames, n_fft//2+1)   ← 复数频谱
↓ |·|²
power      (n_frames, n_fft//2+1)   ← 功率谱
↓ @ mel_filterbank.T
mel_energy (n_frames, n_mels)       ← Mel 能量
↓ log(·+ε)
log_mel    (n_frames, n_mels)       ← 最终特征 ✓
```

**核心公式**（一行）：
$$\text{log\_mel} = \log\!\left(\text{power} \cdot F^T + \varepsilon\right)$$

- `F`：Mel 滤波器组矩阵，shape `(n_mels, n_fft//2+1)`
- `ε = 1e-8`：防止 log(0) = −∞，把最小值钳在 `log(1e-8) ≈ -18`

本节任务：实现 `log_mel_spectrogram(x, sr, n_mels=80, win_len=1024, hop=256)`，与 `aurora.audio.mel.mel_spectrogram` 输出误差 < 1e-9。

## 开课前 2 分钟复习：矩阵乘法就是滤波

- `power @ fb.T` 是把每帧功率谱投影到 Mel 三角滤波器上
- 这里的 `fb` 来自 L46 的三角滤波器组
- 记住 shape：`power (T,F)` × `fb.T (F,M)` → `mel_energy (T,M)`

L21 里你已经见过同一件事的矩阵版本；现在只是在 Audio DSP 里重新出现。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.io import sine, chirp

sr = 16000
# 生成测试信号：2 秒 chirp（200 Hz → 4000 Hz）
x = chirp(200, 4000, duration=2.0, sample_rate=sr)
print(f"x.shape = {x.shape}, sr = {sr}")

## 1. 什么是 log-Mel？

核心公式只有一行：

```
log_mel = log( power_spectrum @ mel_filterbank.T + eps )
```

其中：
- `mel_filterbank` shape `(n_mels, n_fft//2+1)`：每行是一个三角带通滤波器的权重向量
- `power_spectrum` shape `(n_frames, n_fft//2+1)`：每帧的功率谱（|STFT|²）
- `eps = 1e-8`：防止 log(0)，数值保护
- 结果 shape `(n_frames, n_mels)`

**log 压缩的意义**：人耳对响度的感知是对数的（60 dB 和 61 dB 的差异远小于 0 dB 和 1 dB）。不做 log 的 mel 能量跨越多个数量级，神经网络难以学习；log 压缩后数值分布更均匀，梯度更稳定。

---

### 为什么是三角形滤波器，而不是矩形？

**人耳的两个物理事实驱动了这个设计选择**：

1. **非线性频率分辨率**：人耳在低频（< 1 kHz）能分辨相差约 3–5 Hz 的音，在高频（> 4 kHz）需要相差 > 200 Hz 才能感知差异。Mel 尺度把这种非线性编码为：
   ```
   mel = 2595 * log10(1 + Hz / 700)   # 低频密集，高频稀疏
   ```

2. **临界带宽掩蔽**：耳蜗基底膜上，相邻频率的感知细胞有约 50% 的空间重叠；一个强音会"掩蔽"同临界带内的弱音。三角形滤波器通过让相邻滤波器重叠约 50%，实现了与耳蜗物理结构一致的能量混合。

**三角形的数学含义**：越靠近滤波器中心频率，权重越大；向两侧线性衰减至 0。相比矩形（权重 0/1），三角形在边界处平滑过渡，避免了矩形窗引入的频率截断跳变（吉布斯效应在滤波层面的等价物）。

**实际效果**：80 个 Mel bin 覆盖 0–8000 Hz，前 40 个 bin 集中在 0–1730 Hz（语音共振峰最密集的区间，可由 mel_to_hz(np.linspace(0, hz_to_mel(8000), 82)[40]) ≈ 1730 Hz 计算），后 40 个 bin 平摊 1730–8000 Hz——分辨率分配与人耳感知优先级对齐。

In [ ]:
# 演示：log 压缩让能量分布变均匀
from aurora.audio.mel import mel_filterbank, mel_spectrogram
from aurora.audio.stft import stft

S = stft(x, n_fft=1024, hop_length=256)
power = np.abs(S) ** 2
fb = mel_filterbank(80, 1024, sr)
mel_energy = power @ fb.T          # 线性 mel 能量

print(f"线性 mel 能量范围: [{mel_energy.min():.4f}, {mel_energy.max():.4f}]")
print(f"log  mel 能量范围: [{np.log(mel_energy + 1e-8).min():.4f}, {np.log(mel_energy + 1e-8).max():.4f}]")

## 2. Shape 流向

每一步的 shape 变化：

```
x          : (T,)                        # 原始波形，T 个采样点
STFT       : (n_frames, n_fft//2 + 1)   # 复数矩阵
power      : (n_frames, n_fft//2 + 1)   # |STFT|²，实数
fb         : (n_mels,   n_fft//2 + 1)   # Mel 滤波器组
mel_energy : (n_frames, n_mels)          # power @ fb.T
log_mel    : (n_frames, n_mels)          # log(mel_energy + eps)
```

关键转置：`power @ fb.T` 而不是 `fb @ power`。`fb` 的每行是一个 Mel 滤波器（长度 = FFT bins），`power` 的每行是一帧的功率谱，两者做矩阵乘法后每行输出一帧的 n_mels 个能量值。

In [ ]:
# 验证每一步的 shape
print(f"x        : {x.shape}")
print(f"STFT     : {S.shape}")
print(f"power    : {power.shape}")
print(f"fb       : {fb.shape}")
print(f"mel_enrg : {mel_energy.shape}")
print(f"log_mel  : {np.log(mel_energy + 1e-8).shape}")

# 可视化 mel 滤波器组前 5 个滤波器
freqs = np.fft.rfftfreq(1024, 1/sr)
fig, ax = plt.subplots(figsize=(9, 3))
for i in range(5):
    ax.plot(freqs, fb[i], label=f"Mel#{i}")
ax.set_xlabel("频率 (Hz)")
ax.set_ylabel("权重")
ax.set_title("Mel 滤波器组（前 5 个三角滤波器）")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 3. eps 与均值归一化

**eps = 1e-8**：唯一的数值保护。log(0) = -∞ 会让梯度炸掉，加 eps 把最小值钳在 `log(1e-8) ≈ -18.4`，物理上对应极弱的能量，不影响有效帧。

**均值归一化（CMVN）**：训练前通常对整段音频做：

```
log_mel_norm = (log_mel - log_mel.mean(axis=0)) / (log_mel.std(axis=0) + 1e-8)
```

`axis=0` 对每个 Mel bin 独立归一化，消除麦克风频率响应偏差，让模型在不同录音条件下更稳定。Month 2 分类器的预处理层会做这一步，但 `mel_spectrogram` 本身不包含它，保持功能单一。

In [ ]:
# 演示 CMVN 归一化效果
log_mel = np.log(mel_energy + 1e-8)
log_mel_norm = (log_mel - log_mel.mean(axis=0)) / (log_mel.std(axis=0) + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].imshow(log_mel.T, aspect="auto", origin="lower", cmap="magma")
axes[0].set_title("log-Mel（原始）")
axes[0].set_xlabel("帧"); axes[0].set_ylabel("Mel bin")

axes[1].imshow(log_mel_norm.T, aspect="auto", origin="lower", cmap="magma")
axes[1].set_title("log-Mel（CMVN 归一化后）")
axes[1].set_xlabel("帧"); axes[1].set_ylabel("Mel bin")
plt.tight_layout()
plt.show()

print(f"归一化前 均值={log_mel.mean():.3f}  std={log_mel.std():.3f}")
print(f"归一化后 均值={log_mel_norm.mean():.3f}  std={log_mel_norm.std():.3f}")

## 4. ✏️ 实现 `log_mel_spectrogram(x, sr, n_mels=80, win_len=1024, hop=256)`

**三步提示**：

1. 先拿到 STFT，再把复数幅度变成功率谱
2. 用 `mel_filterbank(...)` 把频率轴压到 `n_mels`
3. `np.log(... + 1e-8)` 只是数值保护，不是算法核心

**卡住回**：L44 / L46 / L47

**参考实现**：`solutions/L47_mel_implement_solutions.md`


In [ ]:
# 注意：下方 TODO 使用 stft（已在 cell 4 导入）；aurora_stft 是同一函数的别名，忽略即可
from aurora.audio.stft import stft as aurora_stft
from aurora.audio.mel import mel_filterbank

def log_mel_spectrogram(x, sr, n_mels=80, win_len=1024, hop=256):
    """
    x       : 1-D numpy array, 原始波形
    sr      : 采样率（Hz）
    n_mels  : Mel 滤波器数量，默认 80
    win_len : FFT 窗长（= n_fft），默认 1024
    hop     : 帧移（samples），默认 256
    返回   : (n_frames, n_mels) float64 log-Mel 矩阵
    """
    # ✏️ TODO step 1: STFT → 复数矩阵 S，shape (n_frames, win_len//2+1)

    # ✏️ TODO step 2: power = |S|²，shape 同上

    # ✏️ TODO step 3: mel_filterbank → fb，shape (n_mels, win_len//2+1)

    # ✏️ TODO step 4: mel_energy = power @ fb.T，shape (n_frames, n_mels)

    # ✏️ TODO step 5: return np.log(mel_energy + 1e-8)
    raise NotImplementedError("TODO: 返回 log(mel_filterbank @ magnitude_spectrogram + 1e-8)")

In [ ]:
from aurora.audio.mel import mel_spectrogram

# 构造参考输出：aurora mel_spectrogram 不含 log，手动加 log+eps 作为参考
ref = np.log(mel_spectrogram(x, sr, n_fft=1024, hop_length=256, n_mels=80) + 1e-8)
out = log_mel_spectrogram(x, sr, n_mels=80, win_len=1024, hop=256)

assert out is not None, "函数返回了 None，请检查是否有 return 语句"
assert out.shape == ref.shape, f"shape 不匹配：{out.shape} vs {ref.shape}"
assert np.allclose(out, ref, atol=1e-6), (
    f"数值不匹配，最大误差 = {np.max(np.abs(out - ref)):.2e}"
)
print(f"✅ shape = {out.shape}")
print(f"✅ 最大误差 = {np.max(np.abs(out - ref)):.2e}（< 1e-6）")
print(f"✅ 值域 [{out.min():.2f}, {out.max():.2f}]")

## 5. 参数实验：n_mels = 40 / 80 / 128

**预期现象**：
- `n_mels=40`：频率分辨率粗，热力图垂直方向像素少，低频细节丢失，但计算快；Month 2 小模型可以用
- `n_mels=80`：平衡选择，业界标准（Whisper、wav2vec2 都用 80）
- `n_mels=128`：高频细节更多，对音乐/环境声分类有利，但特征维度增大，模型参数量随之增加

Month 2 实验时可对比三种设定下关键词识别的 val accuracy，通常 80 与 128 差距不大。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, n_mels in zip(axes, [40, 80, 128]):
    feat = log_mel_spectrogram(x, sr, n_mels=n_mels, win_len=1024, hop=256)
    if feat is None:  # stub 未完成时跳过，避免 AttributeError
        ax.set_title(f"n_mels={n_mels}\n（请先完成 cell 10 的 TODO）")
        continue
    im = ax.imshow(feat.T, aspect="auto", origin="lower", cmap="magma")
    ax.set_title(f"n_mels = {n_mels}\nshape {feat.shape}")
    ax.set_xlabel("帧")
    ax.set_ylabel("Mel bin")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle("log-Mel 频谱：n_mels 对比（chirp 200→4000 Hz，2 s）", y=1.02)
plt.tight_layout()
plt.show()

## 本课收束

本节实现了 `log_mel_spectrogram(x, sr, n_mels, win_len, hop)`，输出 `(n_frames, n_mels)` float64 矩阵，与 `aurora.audio.mel.mel_spectrogram` 的参考结果对齐至 1e-6 精度。函数内部依赖 `aurora.audio.stft.stft` 和 `aurora.audio.mel.mel_filterbank`，完整流水线现已固化在 `aurora.audio.mel` 模块中。下一课：**L48** 时频图解析，用多组 mel 图建立线性谱、Mel 谱与对数 Mel 谱的完整视觉记忆。

## ✏️ 闭卷推导检查格 — Mel 滤波器归一化

**规则：关闭上方所有格，仅凭记忆完成以下推导。**

**题目**：三角滤波器 $H_m(k)$ 的定义为：

$$H_m(k) = \begin{cases}
\frac{k - f(m-1)}{f(m) - f(m-1)} & f(m-1) \le k < f(m) \\
\frac{f(m+1) - k}{f(m+1) - f(m)} & f(m) \le k \le f(m+1) \\
0 & \text{otherwise}
\end{cases}$$

1. 写出该滤波器面积（所有 $k$ 的求和）的解析表达式
2. Aurora 的实现是否归一化面积？若是，写出归一化后的表达式

（在此处写推导...）

In [ ]:
# 验证：滤波器矩阵每行之和（面积）是否符合预期
import numpy as np, sys
sys.path.insert(0, 'src')
from aurora.audio.mel import mel_filterbank

fb = mel_filterbank(n_mels=40, n_fft=1024, sample_rate=16000)  # shape (40, 513)
row_sums = fb.sum(axis=1)

# Aurora 使用面积归一化：每行面积 = 1（或接近 1）
assert row_sums.min() > 0, "存在全零行，滤波器构造有误"
cv = row_sums.std() / row_sums.mean()
print(f"滤波器行面积：min={row_sums.min():.4f} max={row_sums.max():.4f} CV={cv:.3f}")
print("✅ 滤波器归一化验证通过")

In [ ]:
# ✏️ 本课自评
l47_review = {
    "pipeline_5steps":         None,  # 记住 STFT→power→@fb.T→log+ε 五步流程？True/False
    "log_mel_implemented":     None,  # log_mel_spectrogram 实现并通过 atol=1e-9？True/False
    "eps_role":                None,  # 理解 ε=1e-8 防 log(0)=-inf，最小值约-18？True/False
    "shape_flow":              None,  # 能手算 (T,)→(n_frames,n_bins)→(n_frames,n_mels) shape？True/False
    "whiteboard_passed":       None,  # 白板推导滤波器归一化闭卷通过？True/False
}

unfilled = [k for k, v in l47_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l47_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L47 全部通关！进入 L48：时频图解（可视化课）')

---

→ **下一课**　[L48 · 时频图解](L48_visual_stft.ipynb)

> 下节课将学习 **时频图解**：线性谱 / Mel 谱 / 对数 Mel 谱三者视觉对比。